In [1]:
import sys
sys.path.insert(0, "..") # dalgebra is here

from dalgebra import *

In [2]:
%pip install --no-build-isolation git+https://github.com/mkauers/ore_algebra.git
%pip install --no-build-isolation -e .

  Cloning https://github.com/mkauers/ore_algebra.git to /tmp/pip-req-build-ggci2xci
  Running command git clone --filter=blob:none --quiet https://github.com/mkauers/ore_algebra.git /tmp/pip-req-build-ggci2xci
  Resolved https://github.com/mkauers/ore_algebra.git to commit d234e3d8ae1d451e734218b02aa5f05a3ffda2d9
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Obtaining file:///home/upm/dalgebra_ore/notebooks
ERROR: file:///home/upm/dalgebra_ore/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Note: you may need to restart the kernel to use updated packages.


In [2]:
B = DifferenceRing(QQ[x], QQ[x].Hom(QQ[x])(x+1)); x = B(x)
S.<z> = DPolynomialRing(B)
P = z[2] - 3*x*z[1] + (x^2 - 1)*z[0]
Q = z[3] - z[0]

In [3]:
R.<x> = QQ[] # base ring
DR = DRing(R, [(x,-1)], types=["skew"])
print(DR)

Differential Ring [[Univariate Polynomial Ring in x over Rational Field], (-d/dx,)]


In [5]:
x = R.gens()[0]
T.<y> = DPolynomialRing(DR)

In [6]:
print(T.operators()[0].twist)

Id


In [7]:
ranking = T.ranking()
ranking.pseudo_quo_rem(x*y[1],y[0])

(1, {(1,): (x, 1)}, 0)

In [8]:
(x*y[2]).order()

2

In [9]:
R.<t> = QQ[]
s = R.Hom(R)([-t])
td = R.derivation_module(twist=s)(1)
tR = DRing(R, td, types=["skew"])
t = tR.gens()[0]

U.<a> = DPolynomialRing(tR)


#print(U.random_element())# Does not work?
print()
for _ in range(10):
    p = tR.random_element()*a[3]+tR.random_element()*a[0]
    q = tR.random_element()*a[1] + tR.random_element()*a[0]
    aa, b, c = U.ranking().pseudo_quo_rem(p,q)
    b = U.ranking().pqr_to_operator(b)
    if b == 0:
        print(p,c)
    else:
        print(aa*p-b.dot(q,a)-c)


0
0
0
0
0
0
0
0
0
0


In [10]:
print((t*a[1]).parent().operators()[0].twist)

Ring endomorphism of Univariate Polynomial Ring in t over Rational Field
  Defn: t |--> -t


In [23]:
def Li_sres_seq(f, g):
    """
    Subresultant computation through Li's PRS algorithm.
    """
    twist = f.parent().operators()[0].twist
    
    def sigma_factorial(x,n,twist=twist):
        if n<=0:
            return 1
        return twist(sigma_factorial(x,n-1))*x
    
    if f.order()<0 or g.order()<0:
        raise NotImplementedError("Only homogeneous subresultants are considered")
    
    # Li does not allow f.order() < g.order(). Compromising for dalgebra.
    if f.order() < g.order():
        f, g = g, f
    
    # 2. Base Case Initialization
    n = f.order()
    m = g.order()
    l_i = n - m
    
    facto_g = sigma_factorial(g.initial(), l_i - 1)
    S_m = facto_g * g
    # 3. S_{m-1}
    _, _, r_first = f.parent().ranking().pseudo_quo_rem(f, g)
    S_m_minus_1 = (-1)**(l_i + 1) * r_first
    if S_m_minus_1.is_zero():
        sres = {}
        for i in range(g.order()):
            sres[i] = f.parent().zero()
        return sres
    sres = {m - 1: S_m_minus_1}
    d = S_m_minus_1.order()
    
    # Subresultant theorem: in between are trivial.
    for index in range(d + 1, m - 1):
        sres[index] = f.parent().zero()
    # Subresultant theorem: next regular can be computed.
    if d < m - 1:
        gap = (m - 1) - d
        num = sigma_factorial(S_m_minus_1.initial(), gap)
        den = sigma_factorial(sigma_factorial(g.initial(),l_i), gap)
        sres[d] = f.parent().base_ring()(num // den) * S_m_minus_1
    
    # To the next subresultant.
    Si, Sj = S_m, S_m_minus_1
    # Trailing leading coefficient instead of regular subresultant.
    Sjplus1_lc = sigma_factorial(g.initial(), l_i)
    for _ in range(n):
        if Sj.is_zero():
            for index in range(Si.order()):
                if index not in sres:
                    sres[index] = 0
            return sres
        si, sj = Si.order(), Sj.order()
        l_i = si - sj
        lc_Si = Si.initial()
        c_i = ((-1)**(l_i + 1)) * sigma_factorial(twist(Sjplus1_lc), l_i) * lc_Si
        
        _, _, r = f.parent().ranking().pseudo_quo_rem(Si, Sj)
        
        # Next subresultant of 1st order, subresultant theorem.
        Sk = r // c_i    
        
        if Sk.is_zero():
            for index in range(sj):
                if index not in sres:
                    sres[index] = f.parent().zero()
            return sres
        
        # The formal index of this subresultant is sj - 1
        Sk_index = sj - 1
        sres[Sk_index] = Sk
        
        d = Sk.order()
        for index in range(d + 1, Sk_index):
            sres[index] = f.parent().zero()
        
        # Compute Sjplus1_lc
        lc_Sj = Sj.initial()
        num_h = sigma_factorial(lc_Sj, l_i)
        den_h = sigma_factorial(Sjplus1_lc, l_i - 1)
        new_Sjplus1_lc = f.parent().base_ring()(num_h // den_h)
        
        # Compute the next regular subresultant
        if d < Sk_index:
            gap = Sk_index - d
            num = sigma_factorial(unofficial_initial(Sk), gap, s)
            den = sigma_factorial(new_Sjplus1_lc, gap, s)
            sres[d] = f.parent().base_ring()(num // den) * Sk
        
        # Next iteration
        Sjplus1_lc = new_Sjplus1_lc
        Si, Sj = Sj, Sk
    return sres

In [21]:
p = t**2*a[2]+t*a[1]
q = -t*a[1]+t*a[0]

In [24]:
import random
# Main random test
R.<t> = QQ[]
s = R.Hom(R)([-t])
td = R.derivation_module(twist=s)(1)
tR = DRing(R, td, types=["skew"])
t = tR.gens()[0]
O.<y> = DPolynomialRing(tR)
gen = O.gens()[0] # Assuming single skew.

n = 7
count = 0
for _ in range(n**2):
    # Generate random polynomials
    p = sum([R.random_element() * gen[i] for i in range(random.randint(1,n))])
    q = sum([R.random_element() * gen[i] for i in range(random.randint(1,n))])
    
    # Handling non-homogeneous attempts.
    if O(p).order() == -1 or O(q).order() == -1:
        try: 
            Li_sres_seq(p,q)
        except NotImplementedError:
            assert True
            continue
        except:
            assert False , f"Wrong exception with p {p} and q {q}" 
    
    Li_dic = Li_sres_seq(p, q)
    sres_dalgebra = O.sylvester_subresultant_sequence(p, q)
    
    # Degree 0 for p and q 
    if Li_dic == {} or sres_dalgebra == ():
        assert Li_dic == {} and sres_dalgebra == ()
        
    for k, sres_d in enumerate(sres_dalgebra):
        sres_L = Li_dic[k]
        
        if sres_d.is_zero():
            assert sres_L.is_zero(), f"Failed at deg {k}: dalgebra is 0, Li_dic is {sres_L},\n p : {p}, \nq : {q}"
        else:
            # Adapted parity between dalgebra and Li.
            if p.order() >= q.order():
                expected_parity = ((p.order() - k) * (q.order() - k)) % 2
                assert sres_L == (-1)**expected_parity*sres_d , f"Failed at deg {k}: \ndal: {sres_d} \nLi : {sres_L} \n p: {p} \n q : {q}"
            else:
                assert sres_L == sres_d , f"Failed at deg {k}: \ndal: {sres_d} \nLi : {sres_L} \n p: {p} \n q : {q}"

In [16]:
Li_sres_seq(p,q)

{0: -(t^4 + t^3)*a_0}